# UK Crime Data – Data Cleaning and Pre-Processing

**Author:** Adam Choy

**Date:** 10th March 2026

**Dataset:** UK Street-Level Crime Dataset

**Police Forces:** Metropolitan Police, Thames Valley, West Midlands, West Yorkshire 

**Time Period:** 2 Years, 1st February 2024 - 31st January 2026

**Steps:** 
1. Import Libraries<br>

2. Load Data into Python<br>

3. Data Inspection <br>

4. Add New Data <br>

5. Fix Data Types and Parse Dates <br>

6. Drop Columns <br>

7. Handle Missing Values <br>

8. Remove Duplicates<br>

9. Final Check<br>

10. Save Data for Analysis

## 1. Import Libraries

Import all necessary libraries required for data cleaning.

In [17]:
import pandas as pd 
import numpy as np
import glob
import os

## 2. Load Data into Python
The CSV data is loaded into Python so it can examined as one combined dataframe.

In [18]:
# CSV files are organised as follows:
'''
UK_Crime_Data/
    ├── 2024-02/
    │   ├── 2024-02-metropolitan-street.csv
    │   └── 2024-02-west-yorkshire-street.csv
    ├── 2024-03/
    │   ├── 2024-03-metropolitan-street.csv
    │   └── ...
    └── ...
'''
# The root folder, UK_Crime_Data contains monthly subfolders, the format YYYY-MM
# Within these monthly folders, there is one CSV per police force
# CSV naming convention: YYYY-MM-force-name-street.csv


#Load all CSVs from monthly subfolders
folder_path = "UK_Crime_Data"
all_files = glob.glob(os.path.join(folder_path, "**", "*.csv"), recursive=True)

df = pd.concat(
    [pd.read_csv(f) for f in all_files],
    ignore_index=True
)

# Total CSV files should be 96, as there are 4 police forces who each have 24 months of data (4 * 24 = 96)
print(f"Total CSV files: {len(all_files)}")
print(f"{df.shape[0]:,} rows")
print(f"{df.shape[1]} columns")

Total CSV files: 97
7,875,206 rows
12 columns


The expected number of CSV files have been loaded in.

## 3. Data Inspection
Examine the structure of the dataset including shape, column names, data types, and previews of the data.

In [3]:
# Preview the beginning of the data
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,Court result unavailable,NaN
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,Unable to prosecute suspect,NaN
2,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
3,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
4,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN


First row loaded is corresponds to a crime that occured in Feburary 2022 that was reported by the Metropolian Police Service. Context column appears to be missing data, and this will be investigated further on.

In [4]:
# Preview the end of the data
df.tail()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
3937598,5aac7a1a203192400f37441e03b37e5036738ef3e3119d...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Investigation complete; no suspect identified,NaN
3937599,f69de500d37c1bc9c244080182058d575be4ddec9bec66...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN
3937600,62c39fb54934c90cec50cfbe8cb9541496470522d29d6b...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN
3937601,e7cd9a2e956a88fa0729a23445745990ebe16db644053e...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN
3937602,4302f75bff4030d28b448875d1aad73d3c45893539bd46...,2026-01,West Yorkshire Police,West Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Under investigation,NaN


Last row loaded in corresponds to a crime that occured in January 2026 that was reported by West Yorkshire Police. Again, context is missing values.

In [5]:
# Understand the structure of the data
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# Look at the column names and data types
print("Column Names and Data Types:")
df.info()

Rows: 3,937,603
Columns: 12
Column Names and Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3937603 entries, 0 to 3937602
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Crime ID               object 
 1   Month                  object 
 2   Reported by            object 
 3   Falls within           object 
 4   Longitude              float64
 5   Latitude               float64
 6   Location               object 
 7   LSOA code              object 
 8   LSOA name              object 
 9   Crime type             object 
 10  Last outcome category  object 
 11  Context                float64
dtypes: float64(3), object(9)
memory usage: 360.5+ MB


There are 3,937,603 rows of data, and 12 columns. The datatypes of all columns are objects, except from the latitute, longitute, and context columns, which are in float format. This is expected for latiture and longiture, but surprising for context, which should be an object.

In [6]:
# Look at the summary statistics of latitute, longiture, and context
df.describe()

,Longitude,Latitude,Context
count,3.923469e+06,3.923469e+06,0.0
mean,-7.344499e-01,5.203935e+01,NaN
std,7.797400e-01,8.204920e-01,NaN
min,-6.789849e+00,4.996553e+01,NaN
25%,-1.561880e+00,5.150096e+01,NaN
50%,-2.857840e-01,5.156163e+01,NaN
75%,-9.903100e-02,5.247788e+01,NaN
max,1.745418e+00,5.573572e+01,NaN


Context column is confirmed to be completely empty, and will need to be removed.

In [7]:
# Check all four police forces have been loaded in using the reported by column
print("Police forces:", df["Reported by"].unique())

Police forces: ['Metropolitan Police Service' 'Thames Valley Police'
 'West Midlands Police' 'West Yorkshire Police']


In [8]:
# Check all four police forces have been loaded in using the reported by column
print("Police forces:", df["Falls within"].unique())

Police forces: ['Metropolitan Police Service' 'Thames Valley Police'
 'West Midlands Police' 'West Yorkshire Police']


All four police forces have correctly been loaded in. The "Falls within" column refers to the geograhpic area the crime occurred in, and the "Reported by" column refers to the police force that reported it. The "Reported by" column will be dropped later on as the geopgrahic area a crime occurred in is more likely to be a determinant of house prices.

In [9]:
# Check types of crime that are reported
print("Crime types:", df["Crime type"].unique())

Crime types: ['Drugs' 'Violence and sexual offences' 'Anti-social behaviour'
 'Other theft' 'Vehicle crime' 'Burglary' 'Criminal damage and arson'
 'Public order' 'Robbery' 'Other crime' 'Shoplifting'
 'Theft from the person' 'Bicycle theft' 'Possession of weapons']


Crime types are more specific than expected. They will benefit from being categorised into groups.

In [10]:
# Check types of crime outcomes
print("Types of crime outcomes:", df["Last outcome category"].unique())

Types of crime outcomes: ['Court result unavailable' 'Unable to prosecute suspect' nan
 'Local resolution' 'Investigation complete; no suspect identified'
 'Status update unavailable' 'Offender given penalty notice'
 'Offender given a caution' 'Action to be taken by another organisation'
 'Formal action is not in the public interest'
 'Further investigation is not in the public interest'
 'Awaiting court outcome' 'Further action is not in the public interest'
 'Suspect charged as part of another case'
 'Offender given a drugs possession warning' 'Under investigation']


In [11]:
print("Number of crime outcome types:", df["Last outcome category"].nunique())

Number of crime outcome types: 15


There are fifteen different outcomes of crime, many of which seem to overlap. They will be catergorised into prosecuted, pending, and unsuccessful.

In [12]:
print(df["Last outcome category"].value_counts())

Last outcome category
Investigation complete; no suspect identified          1512554
Unable to prosecute suspect                            1128257
Under investigation                                     163895
Status update unavailable                               162485
Court result unavailable                                113032
Awaiting court outcome                                   88335
Local resolution                                         77309
Action to be taken by another organisation               36834
Offender given a caution                                 15815
Further investigation is not in the public interest       8848
Further action is not in the public interest              6482
Offender given penalty notice                             4673
Formal action is not in the public interest               4556
Suspect charged as part of another case                    853
Offender given a drugs possession warning                    2
Name: count, dtype: int64


In [13]:
# Check how many reports of burglaries per police force
print("Total reports of bulgary:",df["Crime type"].value_counts()["Burglary"])
df[df["Crime type"] == "Burglary"].groupby("Reported by").size()

Total reports of bulgary: 174102


Reported by
Metropolitan Police Service    99409
Thames Valley Police           14170
West Midlands Police           30992
West Yorkshire Police          29531
dtype: int64

**Inspection Summary:**
- There are about 3.9 million rows of data, indicating 3.9 million reports of street-level crime within four English police forces and within 2 years.
- There are 12 columns, all of which are self-explanatory except LSOA abbreviation
- The geographic average of where the crimes occur is in Milton Keynes, a central UK city halfway between London and Birmingham
- LSOA stands for Lower layer Super Output Area, a small geographic unit containing about 400-1200 households
- Crime types are will benefit by being put into groups
- Crime outcomes will also benefit by being group into "Prosecuted, Pending, or Unsuccessful"
  
- Data is ordered in chronological order, with the oldest 2024 data found in the head and the most recent 2026 data found in the tail
- Context column is always null in street level data. As it does not support the analysis, it will be dropped
- It is redunant to keep both "Reported by" and "Falls within" columns. The Reported by column will be dropped as it is less revelant to house prices
- "Crime ID" is extremely long and complex
- Months are currently in object type, and will be need to be converted to datatime format to allow for time analysis

## 4. Add new columns
Before the cleaning process begins, new data that will aid the anaylsis will be added. The new data will be the population each police force serves and house prices by each LSOA.

**Population**<br>
Population data per police force area was taken from [ONS: Population estimates for police force areas in England and Wales by single year of age and sex, mid-1991 to mid-2024.](https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/adhocs/3194populationestimatesforpoliceforceareasinenglandandwalesbysingleyearofageandsexmid1991tomid2024) 
This data is from mid 2024.

The population data is as following:

**Metropolitan Police:**	9,074,625 <br>
**Thames Valley:**	2,640,201 <br>
**West Midlands:**	3,036,605 <br>
**West Yorkshire:** 2,435,236 <br>

In [19]:
population_data = {
    "Metropolitan Police Service": 9_074_625,
    "Thames Valley Police": 2_640_201,
    "West Midlands Police": 3_036_605,
    "West Yorkshire Police": 2_435_236,
}

As there are only four unique values, 

In [ ]:
population_df = pd.DataFrame(
    list(population_data.items()),
    columns=["Police Force", "Population"]
)

**House Prices**<br>
Population data per police force area was taken from [ONS: Median house prices by lower layer super output area: HPSSA dataset 46](https://www.ons.gov.uk/peoplepopulationandcommunity/housing/datasets/medianpricepaidbylowerlayersuperoutputareahpssadataset46). <br>
This data is from early 2023, the latest release.

In [20]:
# Load house price data
house_prices = pd.read_csv("House_Price_Data.csv")

In [ ]:
# Preview to check column names
house_prices.head()

In [21]:
# Drop LSOA name from house prices before merging (keep only what you need)
house_prices = house_prices[["LSOA code", "House Price"]]  # update to your exact column name

# Now merge
df = pd.merge(df, house_prices, on="LSOA code", how="left")

In [22]:
# Check merge has been successful 
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,House Price
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,Court result unavailable,NaN,"422,500"
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,Unable to prosecute suspect,NaN,"412,500"
2,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,"425,000"
3,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,"425,000"
4,NaN,2024-02,Metropolitan Police Service,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,"425,000"


House prices have been joined correctly after validating with my original house prices file.

**Crime Category**<br>
Crime will be split up into 5 categories: Violent Crime, Property Crime, Anti-Social, Drug Related, and Other.

In [ ]:
crime_categories = {
    # Violent Crime
    "Violence and sexual offences": "Violent Crime",
    "Robbery":                      "Violent Crime",
    "Possession of weapons":        "Violent Crime",

    # Property Crime
    "Burglary":                     "Property Crime",
    "Vehicle crime":                "Property Crime",
    "Other theft":                  "Property Crime",
    "Theft from the person":        "Property Crime",
    "Bicycle theft":                "Property Crime",
    "Shoplifting":                  "Property Crime",
    "Criminal damage and arson":    "Property Crime",

    # Anti-Social & Public Order
    "Anti-social behaviour":        "Anti-Social",
    "Public order":                 "Anti-Social",

    # Drug Related
    "Drugs":                        "Drug Related",

    # Other
    "Other crime":                  "Other",
}

# Map to new column
df["Crime category"] = df["Crime type"].map(crime_categories)

# Check all crime types were mapped
print(df["Crime category"].value_counts())
print(f"\nUnmapped crimes: {df['Crime category'].isnull().sum()}")

In [ ]:
df.head()

In [ ]:
print(f"Shape after merge : {df.shape}")
print(f"Missing house prices: {df['house_price'].isnull().sum():,}")

In [ ]:
# 100% of values in the context column are null
print(f"Total rows: {len(df):,}")
print(f"Null values: {df['Context'].isnull().sum():,}")

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Rows removed: {before - len(df):,}")
print(f"Rows remaining: {len(df):,}")

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
print(df.columns.tolist())

In [ ]:
# Rename columns to snake_case for easier coding
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Renamed columns:")
print(df.columns.tolist())

## 5. Fix Data Types and Parse Dates
Parsing the dates into months and changing data type to datetime format will allow seasonal analysis of crime. 

In [1]:
df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m", errors="coerce")
df["Year"]      = df["Month"].dt.year
df["Month number"] = df["Month"].dt.month
print(df[["Month", "Year", "Month number"]].head(3))

NameError: name 'pd' is not defined

In [ ]:
df.head()

## 6. Drop columns
The following columns need to be dropped:
1. Context
2. Reported By
3. Month
4. Crime Outcome

Before they are dropped, missing values per column are calculated to justify them being dropped.

In [30]:
# Check missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

print("Missing values summary:")
print(missing_df)

Missing values summary:
                       Missing Count  Missing %
Context                      7875206     100.00
Crime ID                     1227346      15.58
Last outcome category        1227346      15.58
House Price                   755286       9.59
Longitude                      28268       0.36
Latitude                       28268       0.36
LSOA code                      28270       0.36
LSOA name                      28270       0.36
Month                              0       0.00
Reported by                        0       0.00
Falls within                       0       0.00
Location                           0       0.00
Crime type                         0       0.00
Year                               0       0.00
Month number                       0       0.00


**Context** <br>
"Context" column is completely empty so it is dropped from the data


In [33]:
df = df.drop(columns=["Context"])

KeyError: "['Context'] not found in axis"

**Reported by** <br>
The reported by column is dropped as the geographic area a crime occurs in is likely to affect house prices more than the police force that reported it. This column is redundant.

In [35]:
df = df.drop(columns = ["Reported by"])

**Month** <br>
Month column has been dropped as it is redunant now that it has been parsed into "Year" and "Month number".

In [37]:
df = df.drop(columns = ["Month"])

**Last outcome category** <br>
"Last outcome category" column is dropped as not useful for analysis. 
Assuming that reports of crime are a determinant of house prices, not the legal outcome of the crime

In [34]:
df = df.drop(columns = ["Last outcome category"])

In [38]:
# Check column drops have been successful
print(df.columns)

Index(['Crime ID', 'Falls within', 'Longitude', 'Latitude', 'Location',
       'LSOA code', 'LSOA name', 'Crime type', 'House Price', 'Year',
       'Month number'],
      dtype='object')


## 7. Handle Missing Values

In [ ]:
# Check total missing values
df.isnull().sum()

In [ ]:
# Check missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

print("Missing values summary:")
print(missing_df[missing_df["Missing Count"] > 0])

In [ ]:
# Coordinates - fill with median
df["Longitude"] = df["Longitude"].fillna(df["Longitude"].median())
df["Latitude"]  = df["Latitude"].fillna(df["Latitude"].median())

In [ ]:
# LSOA - fill unknown
df["LSOA code"] = df["LSOA code"].fillna("Unknown")
df["LSOA name"] = df["LSOA name"].fillna("Unknown")

In [ ]:
# Crime ID - fill ASB records
df["Crime ID"] = df["Crime ID"].fillna("ASB-NO-ID")

In [ ]:
# Drop rows where critical columns are missing
critical_cols = ["month", "crime_type", "reported_by"]

before = len(df)
df = df.dropna(subset=critical_cols)
after = len(df)

print(f"Rows dropped due to missing critical values: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
# Fill non-critical missing values with 'Unknown'
non_critical = ["last_outcome_category", "location", "lsoa_code", "lsoa_name"]

for col in non_critical:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

print("Missing values after filling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 8. Remove Duplicates

In [ ]:
print(df[df["crime_id"].isnull()]["crime_type"].value_counts())

In [ ]:
print(df[df["crime_id"].isnull()]["crime_type"].value_counts())

In [ ]:
df["crime_id"] = df["crime_id"].fillna("ASB-NO-ID")

# Confirm no nulls remain
print(f"Missing Crime IDs remaining: {df['crime_id'].isnull().sum()}")

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Duplicate rows removed: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
print(df.duplicated().sum())

In [ ]:
df[df.duplicated()]

In [41]:
df.head()

,Crime ID,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,House Price,Year,Month number
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,"422,500",2024,2
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,"412,500",2024,2
2,NaN,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,"425,000",2024,2
3,NaN,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,"425,000",2024,2
4,NaN,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,"425,000",2024,2


## 9. Final Check and Inspection of Cleaned Data
Verify the cleaned dataset is complete, correctly formatted, and free of nulls and duplicates before saving. <br>
There should be __ columns, four police services, a data range from 2024 to 2026, and 0 missing values.

In [44]:
print("===== CLEANING SUMMARY =====")
print(f"Final shape : {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Columns     : {df.columns.tolist()}")
print(f"Date range  : {df['Year'].min()} → {df['Year'].max()}")
print(f"Police Forces      : {df['Falls within'].unique()}")
print(f"Missing vals: {df.isnull().sum().sum()}")
print("=============================")
df.head()

===== CLEANING SUMMARY =====
Final shape : 7,875,206 rows, 11 columns
Columns     : ['Crime ID', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'House Price', 'Year', 'Month number']
Date range  : 2024 → 2026
Forces      : ['Metropolitan Police Service' 'Thames Valley Police'
 'West Midlands Police' 'West Yorkshire Police']
Missing vals: 2095708


,Crime ID,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,House Price,Year,Month number
0,19064acc790fb6125fcde7652fd54c0bca0952ec0ab375...,Metropolitan Police Service,0.694489,51.071719,On or near Golden Square,E01024024,Ashford 013E,Drugs,"422,500",2024,2
1,e936c02a9b2af63a455a83c518543f1debbacb6d15456e...,Metropolitan Police Service,0.867904,52.028241,On or near Brook Hall Road,E01029875,Babergh 009A,Violence and sexual offences,"412,500",2024,2
2,NaN,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,"425,000",2024,2
3,NaN,Metropolitan Police Service,0.138830,51.583433,On or near Thatches Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,"425,000",2024,2
4,NaN,Metropolitan Police Service,0.138781,51.589468,On or near Kingston Hill Avenue,E01000027,Barking and Dagenham 001A,Anti-social behaviour,"425,000",2024,2


## 10. Save Cleaned Data for Analysis
Export the cleaned dataset as a new CSV file ready to be loaded into the EDA notebook for the next step: exploratory data analysis.

In [16]:
output_path = os.path.join("UK_Crime_Data", "uk_crime_clean.csv")
df.to_csv(output_path, index=False)

print(f"Clean data saved to: {output_path}")

Clean data saved to: UK_Crime_Data\uk_crime_clean.csv


In [ ]:
df.to_csv(os.path.join("UK_Crime_Data", "uk_crime_clean.csv"), index=False)
print("Cleaned Data has been Saved.")